# 15 - LQR And MPC Control

## What / Why / How

**What we are trying to do:** Study optimal feedback with LQR and short-horizon optimization with a tiny MPC example.

**Why this matters:** Advanced robots often need controllers that balance accuracy, effort, constraints, and future consequences.

**How we will do it:** Compute an LQR gain for a double integrator, simulate feedback, then choose actions by evaluating short candidate futures.

## Goal

Learn two control ideas that appear in serious robotics:

- LQR: optimal feedback for linear systems.
- MPC: repeatedly optimize a short-horizon action sequence.

In [ ]:
import math
import random
import sys
from pathlib import Path

import numpy as np

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

try:
    import matplotlib.pyplot as plt
    HAS_PLOT = True
except ModuleNotFoundError:
    plt = None
    HAS_PLOT = False

np.set_printoptions(precision=3, suppress=True)

def plot_unavailable():
    if not HAS_PLOT:
        print("Install matplotlib to see plots: pip install -r requirements.txt")

In [ ]:
from robotics_mastery.control import dlqr

dt = 0.1
A = np.array([[1, dt], [0, 1]])
B = np.array([[0.5*dt*dt], [dt]])
Q = np.diag([10.0, 1.0])
R = np.array([[0.1]])
K = dlqr(A, B, Q, R)
print("LQR gain:", K)

In [ ]:
def rollout_lqr(x0, steps=80):
    x = np.array(x0, dtype=float)
    rows = []
    for step in range(steps):
        u = float(np.clip(-(K @ x)[0], -3.0, 3.0))
        x = A @ x + B[:, 0] * u
        rows.append((step*dt, x[0], x[1], u))
    return np.array(rows)

lqr_rows = rollout_lqr([3.0, 0.0])
print("final state:", lqr_rows[-1, 1:3])

In [ ]:
def one_step_mpc_action(x, candidates=np.linspace(-3, 3, 31), horizon=12):
    best_cost, best_u = float("inf"), 0.0
    for u0 in candidates:
        sim = x.copy()
        cost = 0.0
        for h in range(horizon):
            u = u0 if h == 0 else float(np.clip(-(K @ sim)[0], -3, 3))
            cost += sim @ Q @ sim + R[0, 0] * u * u
            sim = A @ sim + B[:, 0] * u
        if cost < best_cost:
            best_cost, best_u = cost, u0
    return float(best_u)

x = np.array([3.0, 0.0])
mpc = []
for step in range(80):
    u = one_step_mpc_action(x)
    x = A @ x + B[:, 0] * u
    mpc.append((step*dt, x[0], x[1], u))
mpc = np.array(mpc)
print("MPC final state:", mpc[-1, 1:3])

In [ ]:
if HAS_PLOT:
    plt.figure(figsize=(8, 3))
    plt.plot(lqr_rows[:, 0], lqr_rows[:, 1], label="LQR position")
    plt.plot(mpc[:, 0], mpc[:, 1], label="MPC position", linestyle="--")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
else:
    plot_unavailable()

## Exercises

1. Increase action limits and compare settling time.
2. Change Q to punish velocity more.
3. Add an obstacle penalty to the MPC cost.